# Importation du dataset

In [ ]:
import pandas as pd
df = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb' 
)
print(f"Dimension du dataset : {df.shape}")
print("Aperçu du dataset  : ")
df.sample(5)
df.tail(20)
#df['code_commune'].str.len()
#df['code_departement'].str.len().value_counts()
# Voir tous les codes départements uniques
# df['code_departement'].unique()
df[df['code_departement'].astype(str).str.len() == 3]
df[df['code_departement'] == "988"]
#df[df['code_departement'].astype(str).str.len() == 3]['code_commune'].str.len().value_counts()

mask = (df['code_departement'].astype(str).str.len() == 3) & (df['code_commune'].astype(str).str.len() == 3)
df_dom = df[mask].copy()

df_dom['dept_last'] = df_dom['code_departement'].astype(str).str[-1:]
df_dom['commune_first'] = df_dom['code_commune'].astype(str).str[0]

df_dom[df_dom['dept_last'] != df_dom['commune_first']][['code_departement', 'code_commune', 'dept_last', 'commune_first']].drop_duplicates().head(20)


mask = (df['code_departement'].astype(str).str.len() == 3) & (df['code_commune'].astype(str).str.len() != 3)
df[mask][['code_departement', 'code_commune', 'libelle_commune']].drop_duplicates().head(20)


# 1. Explorations générales

## Question 1 :  Création et mise à jour des variables

Création ou mise à jours des  variables :
- `code_commune` : département + commune
- `candidat` : prénom + nom 

In [ ]:
# 1. Remplacer fr_etranger par 99
df['code_departement'] = df['code_departement'].replace('fr_etranger', '99')

# 2. Construire le code commune
df['code_commune'] = (
    df['code_departement'].astype(str).str[:2]
    + df['code_commune'].astype(str).str[-3:].str.zfill(3)
)

# 3. Correction Wallis-et-Futuna (986) : on prend les 3 chiffres du département
mask_wallis = df['code_departement'].astype(str) == '986'
df.loc[mask_wallis, 'code_commune'] = (
    '986' + df.loc[mask_wallis, 'code_commune'].astype(str).str[-1:].str.zfill(2)
)

# Vérification
df['code_commune'].str.len().value_counts()

In [ ]:
df['code_commune'] = (
    df['code_departement'].astype(str).str.zfill(2)
    + df['code_commune'].astype(str).str[-3:].str.zfill(3)
)
df.tail(5)
#df['code_commune'].str.len()
df['code_departement'].str.len().value_counts()

In [ ]:
df['candidat'] = df['prenom'].astype(str) + ' ' + df['nom'].astype(str)
df.sample(7)

In [ ]:

df[df['prenom'].isna()]['nom'].unique()


candidats = df[df['prenom'].notna()]['candidat'].nunique()

#f"En 2022, il y avait {candidats} candidats à l'élection présidentielle."

In [ ]:
# Votes exprimés uniquement (on exclut les lignes sans prénom)
df_exprimes = df[df['prenom'].notna()]

# Total des voix par candidat
votes_national = df_exprimes.groupby('candidat')['voix'].sum().reset_index()
votes_national.columns = ['Candidat', 'Nombre votes (total)']

# Total général pour calculer les pourcentages
total_exprimes = votes_national['Nombre votes (total)'].sum()

# Score en %
votes_national['Score (% votes exprimés)'] = (votes_national['Nombre votes (total)'] / total_exprimes * 100).round(2)

# Tri décroissant
votes_national = votes_national.sort_values('Nombre votes (total)', ascending=False).reset_index(drop=True)

votes_national

In [ ]:
from great_tables import GT, md, style, loc

from IPython.display import HTML

display(HTML('<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.min.css">'))
(
    GT(votes_national)
    
    .tab_header(
        title=md("**Elections**"),
        subtitle=md(' Résultats du premier tour (<i class="fa fa-calendar-o" aria-hidden="true"></i> <em>10 avril 2022 </em> )')
    )
    
    .tab_source_note(md("**Table 1.** Résultats du premier tour de l'élection présidentielle 2022"))
    # Labels des colonnes

    # Séparateurs de milliers
    .fmt_number(columns="Nombre votes (total)", sep_mark=" ", dec_mark=",", decimals=0)
    # Pourcentage
    .fmt_number(columns="Score (% votes exprimés)", decimals=2, sep_mark=" ", dec_mark=",")
    # Supprimer le striping (tout blanc)
    .tab_style(
        style=style.fill(color="white"),
        locations=loc.body()
    )
    .tab_options(
        row_striping_include_table_body=False,
        row_striping_include_stub=False,
        table_font_names="Times New Roman",
        column_labels_background_color="white",
    )
)

In [ ]:
from utils import charger_fontawesome, beau_tableau

beau_tableau(
    votes_national,
    col_entiers=["Nombre votes (total)"],
    col_pourcentages=["Score (% votes exprimés)"],
    titre="Elections 🇫🇷",
    sous_titre="Résultats du premier tour",
    date="10 avril 2022",
    note="Table 1. Résultats du premier tour de l'élection présidentielle 2022"
)